# Step 6 — LLM Note Parsing & Classification
**Roadmap reference:** LLM text classification for label enrichment

## Goal
Classify each assignment note as:
- **(A) Avoidable reassignment** — could have been prevented with better initial routing
- **(B) Necessary — complexity change** — claim evolved and genuinely required a different adjuster
- **(C) Manual override** — supervisor or adjuster bypassed routing logic
- **(D) Unclear** — insufficient information in the note

## Why this matters
A significant portion of routing intelligence sits in free-text notes that
structured rules completely miss. These labels also enrich the training data
for Step 4's complexity model.

## Sample size
A sample of 50 notes is classified. In a real pilot, manually label 100-200
first as ground truth, then measure LLM agreement rate (target >85%).

In [ ]:
import pandas as pd
import json
import os
import toml
from pathlib import Path
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

DATA = Path("data")
history = pd.read_csv(DATA / "step1_tagged_assignments.csv")

# Sample 50 non-initial events (reassignments only — most interesting for classification)
sample = (history[history["assignment_num"] > 1]
          .sample(50, random_state=42)
          .reset_index(drop=True))

print(f"Sample: {len(sample)} notes")
print("Event type mix:")
print(sample["event_type"].value_counts().to_string())

In [ ]:
# ── Load LLM client ───────────────────────────────────────────────────────────
def _req_secret(key):
    try:
        import streamlit as st
        val = st.secrets.get(key)
        if val: return str(val).strip()
    except Exception:
        pass
    secrets_path = Path(__file__).parents[1] / ".streamlit" / ".secrets.toml"
    if secrets_path.exists():
        cfg = toml.load(str(secrets_path))
        if key in cfg: return str(cfg[key]).strip()
    raise RuntimeError(f"Missing secret: {key}")

llm = AzureChatOpenAI(
    temperature=0,
    azure_endpoint=_req_secret("AZURE_CHAT_ENDPOINT"),
    openai_api_key=_req_secret("AZURE_CHAT_API_KEY"),
    openai_api_version=_req_secret("AZURE_CHAT_API_VERSION"),
    deployment_name=_req_secret("AZURE_CHAT_DEPLOYMENT"),
    model_name=_req_secret("AZURE_CHAT_MODEL"),
    model_kwargs={"response_format": {"type": "json_object"}},
)
print("LLM client ready")

In [ ]:
# ── Batch classify in groups of 10 ────────────────────────────────────────────
SYSTEM_PROMPT = """You are an insurance claims analyst.
Classify each assignment note into exactly one category. Always respond with valid JSON.

Categories:
A = Avoidable reassignment — initial routing was wrong; better upfront data could have prevented this move
B = Necessary — claim genuinely evolved (new exposure, large loss, legal involvement) requiring a different adjuster
C = Manual override — supervisor or adjuster bypassed normal routing logic intentionally
D = Unclear — not enough information in the note to determine

Also extract any adjuster skill or license requirement implied by the note (e.g. "large loss", "SIU", "subrogation").""\"

results = []
batch_size = 10

for batch_start in range(0, len(sample), batch_size):
    batch = sample.iloc[batch_start:batch_start+batch_size]
    notes_text = "\n".join(
        f"Note {i+1}: {row['activity_notes']}"
        for i, (_, row) in enumerate(batch.iterrows())
    )
    user_msg = f"""Classify each of the following {len(batch)} assignment notes.
For each note return: category (A/B/C/D), confidence (high/medium/low), implied_skill (or null).

{notes_text}

Respond as JSON: {{"results": [{{"note": 1, "category": "A", "confidence": "high", "implied_skill": null}}, ...]}}"""

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])
    parsed = json.loads(response.content)
    batch_results = parsed.get("results", [])
    for r in batch_results:
        idx = batch_start + r.get("note", 1) - 1
        if idx < len(sample):
            results.append({
                "claim_id":       sample.iloc[idx]["claim_id"],
                "assignment_num": sample.iloc[idx]["assignment_num"],
                "event_type":     sample.iloc[idx]["event_type"],
                "activity_notes": sample.iloc[idx]["activity_notes"],
                "cause_tag":      sample.iloc[idx]["cause_tag_refined"],
                "category":       r.get("category", "D"),
                "confidence":     r.get("confidence", "low"),
                "implied_skill":  r.get("implied_skill"),
            })
    print(f"  Classified notes {batch_start+1}–{min(batch_start+batch_size, len(sample))}")

df_classified = pd.DataFrame(results)
print(f"\nTotal classified: {len(df_classified)}")

In [ ]:
# ── Results ───────────────────────────────────────────────────────────────────
import plotly.express as px

cat_map = {"A": "Avoidable", "B": "Necessary", "C": "Manual Override", "D": "Unclear"}
df_classified["category_label"] = df_classified["category"].map(cat_map)

print("Classification breakdown:")
print(df_classified["category_label"].value_counts().to_string())
print("\nConfidence breakdown:")
print(df_classified["confidence"].value_counts().to_string())

fig = px.pie(df_classified, names="category_label",
             title="Assignment Note Classification — 50 Sample Notes",
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

df_classified.to_csv(DATA / "step6_llm_classifications.csv", index=False)
print("\nSaved → data/step6_llm_classifications.csv")

In [ ]:
# ── Implied skills extracted ───────────────────────────────────────────────────
skills = (df_classified[df_classified["implied_skill"].notna()]
          [["claim_id", "implied_skill", "category_label"]])
print("Implied skill requirements extracted from notes:")
print(skills.to_string(index=False))